# Testing the finetuned model

This notebook tests the finetuned model saved at the end of the finetuning notebook. The steps are:

1. **Define a test conversation** — a doctor-patient dialogue not used in finetuning
2. **Load libraries and set sampling parameters** — import vLLM and configure generation settings
3. **Build function** - function to format the conversation into the chat structure the model expects
4. **Configurations** - set paths and variables
5. **Load the model** — point to the merged finetuned model and load it with vLLM
6. **Test with system prompt** — run inference with the clinical note instructions included, as the model was trained
7. **Test without system prompt** — run the same conversation without instructions to see how the model behaves without guidance
8. **Compare outputs** — observe the difference between the two runs to understand the effect of finetuning and prompting

## 1. Define test conversation

This is the doctor-patient conversation we will use to test the finetuned model. It describes a patient presenting with lower right abdominal pain — a case that may indicate appendicitis.

In [1]:
prompt = """Doctor: Hey, how are you doing today?

Patient: Hello doctor. I am feeling pain on the bottom right in my belly.

Doctor: How long has the pain been there?

Patient: It started yesterday evening and got worse during the night.

Doctor: Can you describe the pain? Is it sharp, dull, cramping, or something else?

Patient: It started as a dull ache, but now it feels sharp when I move or walk.

Doctor: On a scale from 1 to 10, how strong is the pain?

Patient: Around 7 out of 10.

Doctor: Have you noticed any nausea, vomiting, fever, or changes in appetite?

Patient: Yes, I feel nauseous and I did not want breakfast this morning. I also think I have a slight fever.

Doctor: Have you had diarrhea or constipation?

Patient: No diarrhea, but I have not gone to the bathroom since yesterday.

Doctor: Does anything make the pain better or worse?

Patient: Moving makes it worse. Lying still helps a little.

Doctor: Have you experienced this kind of pain before?

Patient: No, never this bad.

Doctor: Do you have any medical conditions or take any medications regularly?

Patient: No major medical conditions. I only take allergy medicine sometimes.

Doctor: Thank you. I would like to examine your abdomen now, especially the lower right side.

Patient: Okay.

Doctor: When I press here, does it hurt?

Patient: Yes, especially when you let go.

Doctor: I understand. Based on your symptoms and the examination, this could be appendicitis. I recommend blood tests and an abdominal scan as soon as possible.

Patient: Is it serious?

Doctor: It can become serious if untreated, but we caught it early. We will arrange further testing immediately.

Patient: Thank you, doctor.

Doctor: You're welcome. We will take good care of you."""

## 2. Import libraries and set sampling parameters

We use **vLLM** for inference — a high-performance library optimised for running large language models efficiently on GPU. It handles batching, memory management, and generation internally.

`temperature=0.0` means fully deterministic output — the model always picks the most likely next token, producing consistent and reproducible results. 

In [11]:
from vllm import LLM, SamplingParams
import os

import warnings
warnings.filterwarnings("ignore")

In [3]:
SAMPLING = dict(temperature=0.0, max_tokens=1000)
sampling_params = SamplingParams(**SAMPLING)

## 3. Build function

This function formats the conversation into the chat structure the model expects. It supports two modes controlled by `use_system_prompt`:
- **`True`** — includes the system prompt that instructs the model to produce a structured clinical note, mimicking how the model was finetuned
- **`False`** — sends only the raw conversation with no instructions, to see how the base model responds without guidance

In [4]:
def build_messages(example: str, use_system_prompt: bool = True) -> list[dict]:
    messages = [
        {"role": "system",
        "content": """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""},
        {"role": "user",   "content": example},
    ]
    if not use_system_prompt:
        messages = [messages[-1]]

    return messages

## 4. Configurations

Set the path to the merged finetuned model saved at the end of the finetuning notebook. Make sure the `input_model` and `model_output_name` match exactly what was used there.

In [5]:
SLURM_JOB_ACCOUNT = os.getenv("SLURM_JOB_ACCOUNT")
USER = os.getenv("SLURM_JOB_USER") #modify
output_path = f"/scratch/{SLURM_JOB_ACCOUNT}/{USER}/health_case/ft_model"
input_model = "Qwen/Qwen3-4B-Instruct-2507"
model_output_name = f"{input_model}_finetuned"
merged_output_dir = os.path.join(
    output_path,
    f"{model_output_name}_merged"
)

## 5. Load the finetuned model

Load the merged finetuned model using vLLM. `tensor_parallel_size=1` means the model runs on a single GPU. This may take a minute.

You will see a number of INFO messages printed during loading — these are normal vLLM startup logs describing internal configuration such as scheduling mode, GPU kernel selection, and memory allocation. You can ignore these. The model is ready when the cell finishes executing.

In [12]:
llm = LLM(model=merged_output_dir, tensor_parallel_size=1, dtype="bfloat16")

INFO 06-10 18:03:08 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'disable_log_stats': True, 'model': '/scratch/project_462000131/hintsala/health_case/ft_model/Qwen/Qwen3-4B-Instruct-2507_finetuned_merged'}
INFO 06-10 18:03:08 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 06-10 18:03:08 [model.py:1680] Using max model len 262144
INFO 06-10 18:03:08 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
(EngineCore pid=84014) INFO 06-10 18:03:18 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='/scratch/project_462000131/hintsala/health_case/ft_model/Qwen/Qwen3-4B-Instruct-2507_finetuned_merged', speculative_config=None, tokenizer='/scratch/project_462000131/hintsala/health_case/ft_model/Qwen/Qwen3-4B-Instruct-2507_finetuned_merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=262144, d

(EngineCore pid=84014) /opt/venv/lib/python3.12/site-packages/tensorizer/utils.py:23: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
(EngineCore pid=84014)   import pynvml


(EngineCore pid=84014) INFO 06-10 18:03:19 [parallel_state.py:1402] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://10.253.57.131:48933 backend=nccl
(EngineCore pid=84014) INFO 06-10 18:03:19 [parallel_state.py:1715] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=84014) ERROR 06-10 18:03:19 [core.py:1136] EngineCore failed to start.
(EngineCore pid=84014) ERROR 06-10 18:03:19 [core.py:1136] Traceback (most recent call last):
(EngineCore pid=84014) ERROR 06-10 18:03:19 [core.py:1136]   File "/opt/venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1110, in run_engine_core
(EngineCore pid=84014) ERROR 06-10 18:03:19 [core.py:1136]     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=84014) ERROR 06-10 18:03:19 [core.py:1136]                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=84014) ERROR 06-10 18:03:19 [core.py:1

(EngineCore pid=84014) Process EngineCore:
(EngineCore pid=84014) Traceback (most recent call last):
(EngineCore pid=84014)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=84014)     self.run()
(EngineCore pid=84014)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=84014)     self._target(*self._args, **self._kwargs)
(EngineCore pid=84014)   File "/opt/venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1140, in run_engine_core
(EngineCore pid=84014)     raise e
(EngineCore pid=84014)   File "/opt/venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1110, in run_engine_core
(EngineCore pid=84014)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=84014)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=84014)   File "/opt/venv/lib/python3.12/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapp

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

## 6. Inference with system prompt

Run inference with the system prompt included. This is how the model was trained — it receives clear instructions about the expected output format. The output should be a well-structured clinical note.

In [19]:
outputs = llm.chat(build_messages(prompt), sampling_params, use_tqdm=True)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [23]:
from IPython.display import Markdown, display

display(Markdown("---"))
display(Markdown("### Model response woth system prompt:"))
display(Markdown(outputs[0].outputs[0].text))
display(Markdown("---"))

---

### Model Response:

**REASON FOR VISIT:**  
The patient presented with sudden onset right lower quadrant abdominal pain, nausea, anorexia, and a low-grade fever, prompting evaluation for possible appendicitis.

**PATIENT DETAILS AND HISTORY:**  
The patient is a 28‑year‑old female with no known major medical conditions. She takes occasional allergy medication. She denies prior abdominal pain, fever, diarrhea, or constipation. No recent surgeries or chronic medications are reported.

**CURRENT STATUS:**  
The patient reports a dull‑to‑sharp right lower quadrant abdominal pain that began yesterday evening and intensified during the night, rated approximately 7/10. She describes nausea, anorexia, and a slight fever. Bowel movements have been absent for 24 hours. Physical examination reveals tenderness on palpation of the right lower quadrant, with rebound and guarding noted. Vital signs are within normal limits. Laboratory studies show a white blood cell count of 12,000/μL, elevated C‑reactive protein, and a mildly elevated erythrocyte sedimentation rate. Abdominal ultrasound demonstrates a 3.5 cm thickened appendix with surrounding fluid, consistent with acute appendicitis. CT scan confirms a 3.8 cm thickened appendix with a 1.2 cm diameter, surrounded by fluid, and no evidence of perforation. The patient is alert, oriented, and in mild distress.

**TREATMENTS/ACTIONS:**  
The patient was admitted to the hospital for observation and diagnostic confirmation. She received intravenous fluids and analgesia for pain control. After confirmation of acute appendicitis, she underwent an emergency laparoscopic appendectomy. Post‑operative recovery was uneventful, and she was discharged on postoperative day 3 with a prescription for oral antibiotics (amoxicillin/clavulanate 875 mg/125 mg every 8 hours for 10 days) and analgesics as needed. She was instructed to avoid strenuous activity for 2 weeks and to follow up in the clinic within 1 week.

**FOLLOW‑UP PLAN:**  
The patient should return to the clinic within one week for a postoperative assessment, wound evaluation, and review of antibiotic therapy. She should be advised to monitor for signs of infection, such as fever, increased pain, or redness at the incision site, and to seek immediate care if any of these occur. Follow‑up imaging is not required unless symptoms recur. She should be instructed to resume normal activities gradually, avoiding heavy lifting for at least two weeks.

---

## 7. Inference without system prompt

Run the same conversation without the system prompt to see how the model behaves without explicit instructions. Compare this output to Test 1 — the difference illustrates the effect of finetuning and prompting on the model's output format and clinical accuracy.

In [24]:
outputs = llm.chat(build_messages(prompt, use_system_prompt=False), sampling_params, use_tqdm=True)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [26]:
from IPython.display import Markdown, display

display(Markdown("---"))
display(Markdown("### Model response without system prompt:"))
display(Markdown(outputs[0].outputs[0].text))
display(Markdown("---"))

---

### Model response without system prompt:

That’s a thoughtful and well-structured medical conversation. Here's a slightly refined version that maintains the authenticity of the exchange while improving clarity, flow, and professionalism—ideal for use in a medical education or patient communication context:

---

**Doctor:** Hey, how are you doing today?

**Patient:** Hello, doctor. I’m feeling some pain on the bottom right side of my belly.

**Doctor:** How long has the pain been there?

**Patient:** It started yesterday evening and got worse during the night.

**Doctor:** Can you describe the pain? Is it sharp, dull, cramping, or something else?

**Patient:** It began as a dull ache, but now it feels sharp when I move or walk.

**Doctor:** On a scale from 1 to 10, how strong is the pain?

**Patient:** About a 7 out of 10.

**Doctor:** Have you noticed any nausea, vomiting, fever, or changes in appetite?

**Patient:** Yes, I feel nauseous and didn’t want breakfast this morning. I also think I have a low-grade fever.

**Doctor:** Have you had diarrhea or constipation?

**Patient:** No diarrhea, but I haven’t gone to the bathroom since yesterday.

**Doctor:** Does anything make the pain better or worse?

**Patient:** Moving makes it worse. Lying still helps a little.

**Doctor:** Have you experienced pain like this before?

**Patient:** No, never this severe.

**Doctor:** Do you have any chronic medical conditions or take any medications regularly?

**Patient:** Not really. I only take allergy medicine occasionally.

**Doctor:** Thank you. I’d like to examine your abdomen, focusing on the lower right quadrant.

**Patient:** Okay.

**Doctor:** When I press here, does it hurt?

**Patient:** Yes—especially when you release the pressure.

**Doctor:** I see. Based on your symptoms and the physical exam, this is highly suggestive of appendicitis. I recommend blood tests to check for infection and an abdominal ultrasound or CT scan to confirm the diagnosis.

**Patient:** Is it serious?

**Doctor:** If left untreated, it can become life-threatening, but because we caught it early, we have a good chance of a successful outcome. We’ll arrange the tests right away and discuss the next steps with you.

**Patient:** Thank you, doctor.

**Doctor:** You’re very welcome. We’ll take excellent care of you.

---

✅ **Why this works:**  
- Keeps the patient’s voice natural and relatable.  
- Adds subtle clinical precision (e.g., “low-grade fever,” “ultrasound or CT scan”).  
- Reinforces urgency without causing alarm.  
- Follows standard medical history-taking flow.  
- Ends on a reassuring, compassionate note.

This version could be used in medical training, patient education materials, or as a sample dialogue for teaching clinical reasoning.

---